# Notebook 03 — Benchmark Circuit Evaluation

Runs the four benchmark circuits (VQE-H2, VQE-LiH, QPE-12, Grover-10) under
three noise profiles (Superconducting, Trapped-ion, Adversarial) across all
four pipeline configurations (Baseline, Mapper-only, QED-only, Joint).

**Prerequisites:** `00_setup_and_data_generation.ipynb` and
`02_scheduler_training.ipynb` must have been executed first so that
`data/training/model_checkpoint.pkl` exists.

In [ ]:
import sys, os, json, time, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from pathlib import Path

from qiskit import QuantumCircuit
from qiskit.qasm3 import loads as qasm3_loads

from src.compiler.mapping_pass import HardwareAwareMappingPass
from src.compiler.cost_computation import NoiseCostFunction
from src.scheduler.scheduler import QEDScheduler
from src.qed.circuit_builder import QEDCircuitBuilder
from src.simulation.simulator import DensityMatrixSimulator
from src.simulation.noise_models import load_noise_model
from src.metrics.aggregator import MetricsAggregator
from src.metrics.statistical_tests import wilcoxon_test, bootstrap_ci
from src.utils.io import load_json, save_results

SEED = 42
np.random.seed(SEED)
N_TRIALS = 100
DATA_DIR = Path('../data')
RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
QASM_DIR = Path('../circuits')

print('Environment ready.')

## 1. Load Circuits

In [ ]:
CIRCUIT_FILES = {
    'VQE-H2':   QASM_DIR / 'vqe_h2.qasm',
    'VQE-LiH':  QASM_DIR / 'vqe_lih.qasm',
    'QPE-12':   QASM_DIR / 'phase_est_12.qasm',
    'Grover-10': QASM_DIR / 'grover_10.qasm',
}

circuits = {}
for name, path in CIRCUIT_FILES.items():
    with open(path) as f:
        qasm_src = f.read()
    circuits[name] = qasm3_loads(qasm_src)
    print(f'{name:12s}  qubits={circuits[name].num_qubits:3d}  '
          f'depth={circuits[name].depth():4d}  '
          f'gates={circuits[name].size():4d}')

## 2. Load Noise Models

In [ ]:
NOISE_FILES = {
    'Superconducting': '../noise_models/ibm_lagos.json',
    'Trapped-ion':     '../noise_models/trapped_ion.json',
    'Adversarial':     '../noise_models/adversarial.json',
}

noise_models = {}
for profile, fpath in NOISE_FILES.items():
    noise_models[profile] = load_noise_model(fpath)
    print(f'Loaded noise model: {profile}')

## 3. Load Trained Scheduler

In [ ]:
import pickle
ckpt_path = DATA_DIR / 'training' / 'model_checkpoint.pkl'
with open(ckpt_path, 'rb') as f:
    checkpoint = pickle.load(f)

scheduler = QEDScheduler(
    model_ds=checkpoint['model_ds'],
    model_r=checkpoint['model_r'],
    lambda_param=0.8,
    mu_param=5e-4,
)
print('Scheduler loaded from checkpoint.')
print(f'  M_ΔS  n_estimators={checkpoint["model_ds"].n_estimators}')
print(f'  M_R   n_estimators={checkpoint["model_r"].n_estimators}')

## 4. Instantiate Pipeline Components

In [ ]:
# SA / ILP mapping-pass hyperparameters (§5 of the paper)
MAPPER_CONFIG = dict(
    T0=1.0,
    alpha=0.995,
    n_iter=5000,
    ilp_window=10,
    latency_budget_ms=2000,
    latency_penalty=1.0,
)

qed_builder = QEDCircuitBuilder()
simulator   = DensityMatrixSimulator(use_gpu=False)  # set True when A100 available
aggregator  = MetricsAggregator(n_bootstrap=10_000, seed=SEED)

print('All pipeline components instantiated.')

## 5. Helper: Run One Configuration

In [ ]:
def run_configuration(
    circuit: QuantumCircuit,
    noise_model,
    device_props: dict,
    mode: str,           # 'baseline' | 'mapper_only' | 'qed_only' | 'joint'
    n_trials: int = N_TRIALS,
    seed: int = SEED,
) -> dict:
    """
    Execute a circuit under a given mode and return aggregated metrics.

    Returns
    -------
    dict with keys: success_probs, retention_fracs, latencies_ms
    """
    rng = np.random.default_rng(seed)
    success_list, retention_list, latency_list = [], [], []

    for trial in range(n_trials):
        t0 = time.perf_counter()

        # --- Compilation ---
        if mode in ('baseline', 'qed_only'):
            # Standard SABRE — use Qiskit's built-in transpiler at O2
            from qiskit import transpile
            from qiskit.providers.fake_provider import GenericBackendV2
            n_q = circuit.num_qubits
            backend = GenericBackendV2(num_qubits=max(n_q, 5))
            compiled = transpile(
                circuit, backend=backend,
                optimization_level=2,
                seed_transpiler=seed + trial,
            )
        else:
            # Hardware-aware SA+ILP mapping pass
            mapper = HardwareAwareMappingPass(
                device_props=device_props,
                **MAPPER_CONFIG,
                seed=seed + trial,
            )
            compiled, _ = mapper.run(circuit)

        # --- QED insertion ---
        retention = 1.0
        if mode in ('qed_only', 'joint'):
            schedule = scheduler.predict(compiled, device_props)
            compiled_qed, retention = qed_builder.insert(
                compiled,
                freq=schedule['freq'],
                positions=schedule['positions'],
            )
        else:
            compiled_qed = compiled

        t_compile = time.perf_counter() - t0

        # --- Simulation ---
        t1 = time.perf_counter()
        result = simulator.run(
            compiled_qed,
            noise_model=noise_model,
            shots=1024,
            seed=int(rng.integers(0, 2**31)),
        )
        t_sim = time.perf_counter() - t1

        success_list.append(result['success_prob'])
        retention_list.append(retention)
        latency_list.append((t_compile + t_sim) * 1000)  # ms

    return {
        'success_probs':    np.array(success_list),
        'retention_fracs':  np.array(retention_list),
        'latencies_ms':     np.array(latency_list),
    }

## 6. Main Benchmark Loop

In [ ]:
MODES = ['baseline', 'mapper_only', 'qed_only', 'joint']
PROFILES = ['Superconducting', 'Trapped-ion', 'Adversarial']
CIRCUITS = list(circuits.keys())

raw_results = {}   # raw_results[(circuit, profile, mode)] = metrics dict

for cname in CIRCUITS:
    circ = circuits[cname]
    for profile in PROFILES:
        nm = noise_models[profile]
        device_props = load_json(NOISE_FILES[profile])
        for mode in MODES:
            key = (cname, profile, mode)
            print(f'Running  {cname:12s} | {profile:16s} | {mode:12s} ... ', end='', flush=True)
            t_start = time.time()
            raw_results[key] = run_configuration(circ, nm, device_props, mode)
            elapsed = time.time() - t_start
            S_mean = raw_results[key]['success_probs'].mean()
            R_mean = raw_results[key]['retention_fracs'].mean()
            print(f'S={S_mean:.3f}  R={R_mean:.3f}  [{elapsed:.1f}s]')

print('\n✓ All benchmark runs complete.')

## 7. Statistical Tests and CI Computation

In [ ]:
rows = []
for cname in CIRCUITS:
    for profile in PROFILES:
        base_S = raw_results[(cname, profile, 'baseline')]['success_probs']
        for mode in MODES:
            m = raw_results[(cname, profile, mode)]
            S_arr = m['success_probs']
            R_arr = m['retention_fracs']
            L_arr = m['latencies_ms']

            S_ci  = bootstrap_ci(S_arr, n_boot=10_000, seed=SEED)
            R_ci  = bootstrap_ci(R_arr, n_boot=10_000, seed=SEED)
            p_val = wilcoxon_test(base_S, S_arr) if mode != 'baseline' else float('nan')

            rows.append({
                'Circuit':    cname,
                'Profile':    profile,
                'Mode':       mode,
                'S_mean':     round(S_arr.mean(), 4),
                'S_ci_lo':    round(S_ci[0], 4),
                'S_ci_hi':    round(S_ci[1], 4),
                'R_mean':     round(R_arr.mean(), 4),
                'R_ci_lo':    round(R_ci[0], 4),
                'R_ci_hi':    round(R_ci[1], 4),
                'Latency_ms': round(np.median(L_arr), 1),
                'p_value':    round(p_val, 4) if not np.isnan(p_val) else 'NA',
                'Significant': 'yes' if (not np.isnan(p_val) and p_val < 0.01) else ('no' if not np.isnan(p_val) else 'NA'),
            })

df_results = pd.DataFrame(rows)
print(df_results[df_results['Profile']=='Superconducting'].to_string(index=False))

## 8. Save Results

In [ ]:
out_path = RESULTS_DIR / 'benchmark_summary.csv'
df_results.to_csv(out_path, index=False)
print(f'Saved → {out_path}')

# Save ablation subset (Superconducting profile only) for notebook 05
df_abl = df_results[df_results['Profile'] == 'Superconducting'].copy()
df_abl.to_csv(RESULTS_DIR / 'ablation_results.csv', index=False)
print(f'Saved → {RESULTS_DIR / "ablation_results.csv"}')

## 9. Quick Sanity Table (Superconducting, all circuits, joint vs baseline)

In [ ]:
sc = df_results[df_results['Profile'] == 'Superconducting']
pivot = sc.pivot_table(
    index='Circuit',
    columns='Mode',
    values='S_mean',
)[MODES]
print('Mean Success Probability — Superconducting Profile')
print(pivot.to_string())